# PrimeKG — Dataset Acquisition and Initial EDA

CSAI 491 Senior Project · AI-Powered Drug Repurposing Map from Biomedical Literature

This notebook downloads PrimeKG (Chandak et al., 2023) and explores its size, structure and quality across **all diseases**.
Run all cells from top to bottom (Runtime → Run all). The download is about 1 GB the first time.

In [ ]:
from pathlib import Path
import shutil
import urllib.request
import pandas as pd
import matplotlib.pyplot as plt

DATA_URL = "https://dataverse.harvard.edu/api/access/datafile/6180620"  # PrimeKG kg.csv (Harvard Dataverse)
DATA_PATH = Path("data/kg.csv")
EXPECTED_COLUMNS = ["relation", "display_relation", "x_index", "x_id", "x_type", "x_name", "x_source",
                    "y_index", "y_id", "y_type", "y_name", "y_source"]
pd.set_option("display.max_rows", 50)

## 1. Acquire the data

In [ ]:
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
if not DATA_PATH.exists():
    # Dataverse returns 403 to Python's default user agent, so identify as a browser.
    request = urllib.request.Request(DATA_URL, headers={"User-Agent": "Mozilla/5.0"})
    partial = DATA_PATH.with_suffix(".part")
    with urllib.request.urlopen(request) as response, open(partial, "wb") as f:
        shutil.copyfileobj(response, f)
    partial.rename(DATA_PATH)
print(f"File size: {DATA_PATH.stat().st_size / 1e9:.2f} GB")

In [ ]:
kg = pd.read_csv(DATA_PATH, dtype=str, low_memory=False)
missing_cols = set(EXPECTED_COLUMNS) - set(kg.columns)
if missing_cols:
    raise ValueError(f"Unexpected file format, missing columns: {missing_cols}")
kg.head()

## 2. Size and structure

In [ ]:
print(f"Rows (directed edges): {len(kg):,}")
print(f"Columns: {kg.shape[1]}")
print(f"Memory in use: {kg.memory_usage(deep=True).sum() / 1e9:.2f} GB")

PrimeKG stores each link in both directions (A→B and B→A). We count **unique links** by keeping one copy of each pair per relation.

In [ ]:
a = kg["x_index"].where(kg["x_index"] < kg["y_index"], kg["y_index"])
b = kg["y_index"].where(kg["x_index"] < kg["y_index"], kg["x_index"])
unique_links = pd.DataFrame({"relation": kg["relation"], "a": a, "b": b}).drop_duplicates()
print(f"Unique links: {len(unique_links):,}")

### 2.1 Nodes by type

In [ ]:
nodes = pd.concat([
    kg[["x_index", "x_type", "x_name"]].set_axis(["index", "type", "name"], axis=1),
    kg[["y_index", "y_type", "y_name"]].set_axis(["index", "type", "name"], axis=1),
]).drop_duplicates("index")
node_counts = nodes["type"].value_counts()
print(f"Total nodes: {len(nodes):,}")
node_counts.to_frame("nodes")

In [ ]:
ax = node_counts.sort_values().plot.barh(figsize=(8, 4), color="black")
ax.set_xlabel("Number of nodes"); ax.set_title("PrimeKG nodes by type")
plt.tight_layout(); plt.savefig("nodes_by_type.png", dpi=200); plt.show()

### 2.2 Links by relation type

In [ ]:
relation_counts = unique_links["relation"].value_counts()
print(f"Relation types: {relation_counts.size}")
relation_counts.to_frame("unique links")

In [ ]:
ax = relation_counts.sort_values().plot.barh(figsize=(8, 7), color="black", logx=True)
ax.set_xlabel("Unique links (log scale)"); ax.set_title("PrimeKG links by relation type")
plt.tight_layout(); plt.savefig("links_by_relation.png", dpi=200); plt.show()

## 3. Drug–disease links (the links our model will learn to predict)

In [ ]:
dd = kg[(kg["x_type"] == "drug") & (kg["y_type"] == "disease")].drop_duplicates()
if dd.empty:
    raise ValueError("No drug-disease links found: check node type names")
dd_counts = dd["relation"].value_counts()
dd_counts.to_frame("drug-disease links")

In [ ]:
treats = dd[dd["relation"] != "contraindication"]  # indication + off-label use
all_diseases = nodes.loc[nodes["type"] == "disease", "index"]
all_drugs = nodes.loc[nodes["type"] == "drug", "index"]
treated = treats["y_index"].unique()
used = treats["x_index"].unique()

summary = pd.Series({
    "Diseases in the graph": len(all_diseases),
    "Diseases with at least one known treatment": len(treated),
    "Diseases with no known treatment": len(all_diseases) - len(treated),
    "Drugs in the graph": len(all_drugs),
    "Drugs with at least one known use": len(used),
})
summary.to_frame("count")

In [ ]:
per_disease = treats.groupby("y_index").size()
ax = per_disease.plot.hist(bins=50, figsize=(8, 4), color="black", logy=True)
ax.set_xlabel("Known treatments per disease"); ax.set_ylabel("Number of diseases (log scale)")
ax.set_title("How many treatments each treated disease has")
plt.tight_layout(); plt.savefig("treatments_per_disease.png", dpi=200); plt.show()
per_disease.describe()

In [ ]:
top_drugs = treats.groupby("x_name").size().nlargest(10).to_frame("diseases treated")
top_drugs

## 4. Data quality

In [ ]:
quality = pd.Series({
    "Missing values (all columns)": int(kg.isna().sum().sum()),
    "Exact duplicate rows": int(kg.duplicated().sum()),
    "Self-loops (node linked to itself)": int((kg["x_index"] == kg["y_index"]).sum()),
    "Rows with an empty node name": int(kg[["x_name", "y_name"]].isna().any(axis=1).sum()),
})
quality.to_frame("count")

In [ ]:
kg.isna().sum().to_frame("missing values")

## 5. Connectivity
How many links each node has (its degree). A few very connected nodes and many sparse ones is typical of biomedical graphs.

In [ ]:
degree = pd.concat([unique_links["a"], unique_links["b"]]).value_counts()
ax = degree.plot.hist(bins=100, figsize=(8, 4), color="black", logy=True)
ax.set_xlabel("Links per node"); ax.set_ylabel("Number of nodes (log scale)")
ax.set_title("Node degree distribution")
plt.tight_layout(); plt.savefig("degree_distribution.png", dpi=200); plt.show()
degree.describe()

## 6. Summary for the report

In [ ]:
report = pd.Series({
    "Rows in kg.csv (directed)": f"{len(kg):,}",
    "Unique links": f"{len(unique_links):,}",
    "Nodes": f"{len(nodes):,}",
    "Node types": node_counts.size,
    "Relation types": relation_counts.size,
    "Drug-disease links": f"{len(dd):,}",
    "Diseases": f"{summary['Diseases in the graph']:,}",
    "Diseases with no known treatment": f"{summary['Diseases with no known treatment']:,}",
    "Missing values": quality["Missing values (all columns)"],
    "Duplicate rows": quality["Exact duplicate rows"],
})
report.to_frame("value")